In [1]:
import mysql.connector

#KONEKSI DATABASE

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="fuzzy_modul1"
)

print("Koneksi database berhasil!")

#FUNGSI KEANGGOTAAN

def fungsi_trapesium_up(x, b_bawah, b_atas):
    """
    Fungsi keanggotaan trapesium bagian naik.

    Persamaan:
    μ(x) = (x - b_bawah) / (b_atas - b_bawah)
    """

    return (x - b_bawah) / (b_atas - b_bawah)


def fungsi_trapesium_down(x, b_bawah, b_atas):
    """
    Fungsi keanggotaan trapesium bagian turun.

    Persamaan:
    μ(x) = (b_atas - x) / (b_atas - b_bawah)
    """

    return (b_atas - x) / (b_atas - b_bawah)

#DAFTAR FUNGSI

fungsi = {
    "trapesium_up": fungsi_trapesium_up,
    "trapesium_down": fungsi_trapesium_down
}

#FUNGSI FUZZIFIKASI

def fuzzifikasi_usia(x, nama_tabel):

    cursor = db.cursor()

    query = f"""
        SELECT b_bawah, b_atas, fungsi
        FROM {nama_tabel}
        WHERE %s >= b_bawah
        AND %s <= b_atas
        ORDER BY b_bawah
    """

    cursor.execute(query, (x, x))

    data = cursor.fetchall()

    cursor.close()

    if len(data) == 0:
        return None

    # Periksa setiap interval yang ditemukan
    for baris in data:

        b_bawah, b_atas, nama_fungsi = baris

        # Jika database memberikan nilai 0
        if nama_fungsi == "0":
            return b_bawah, b_atas, nama_fungsi, 0.0

        # Jika database memberikan nilai 1
        if nama_fungsi == "1":
            return b_bawah, b_atas, nama_fungsi, 1.0

        # Jika database memberikan nama fungsi
        if nama_fungsi in fungsi:

            fungsi_y = fungsi[nama_fungsi]

            nilai = fungsi_y(
                x,
                b_bawah,
                b_atas
            )

            return (
                b_bawah,
                b_atas,
                nama_fungsi,
                round(nilai, 4)
            )

    raise ValueError(
        f"Fungsi '{data[0][2]}' belum dibuat di Python."
    )


#DATA PENGUJIAN

pengujian = [
    ("Bayi", 2, "tb_domain_usia_bayi"),
    ("Bayi", 4, "tb_domain_usia_bayi"),

    ("Anak", 6, "tb_domain_usia_anak"),
    ("Anak", 9, "tb_domain_usia_anak"),

    ("Remaja", 13, "tb_domain_usia_remaja"),
    ("Remaja", 16, "tb_domain_usia_remaja"),

    ("Pemuda", 18, "tb_domain_usia_pemuda"),
    ("Pemuda", 22, "tb_domain_usia_pemuda"),

    ("Dewasa", 35, "tb_domain_usia_dewasa"),
    ("Dewasa", 60, "tb_domain_usia_dewasa"),

    ("Lansia", 80, "tb_domain_usia_lanjut"),
    ("Lansia", 90, "tb_domain_usia_lanjut")
]


#PROSES FUZZIFIKASI

hasil_pengujian = []

for kategori, usia, nama_tabel in pengujian:

    hasil = fuzzifikasi_usia(
        usia,
        nama_tabel
    )

    if hasil is not None:

        b_bawah, b_atas, nama_fungsi, nilai = hasil

        hasil_pengujian.append(
            (
                kategori,
                usia,
                b_bawah,
                b_atas,
                nama_fungsi,
                nilai
            )
        )


#MENAMPILKAN HASIL

print()
print("------------------------------------------")
print("HASIL FUZZIFIKASI USIA")
print("------------------------------------------")

print(
    f"{'No':<4}"
    f"{'Kategori':<10}"
    f"{'Usia':<8}"
    f"{'Interval':<15}"
    f"{'Fungsi':<20}"
    f"{'μ(x)':<8}"
)

print("-" * 75)


for nomor, hasil in enumerate(hasil_pengujian, start=1):

    kategori, usia, b_bawah, b_atas, nama_fungsi, nilai = hasil

    interval = f"{b_bawah}-{b_atas}"

    print(
        f"{nomor:<4}"
        f"{kategori:<10}"
        f"{usia:<8}"
        f"{interval:<15}"
        f"{nama_fungsi:<20}"
        f"{nilai:<8.3f}"
    )


print("----------------------------------------------------------------------------")


#MENUTUP DATABASE

db.close()

Koneksi database berhasil!

------------------------------------------
HASIL FUZZIFIKASI USIA
------------------------------------------
No  Kategori  Usia    Interval       Fungsi              μ(x)    
---------------------------------------------------------------------------
1   Bayi      2       0-3            trapesium_up        0.667   
2   Bayi      4       3-5            trapesium_down      0.500   
3   Anak      6       5-9            trapesium_up        0.250   
4   Anak      9       5-9            trapesium_up        1.000   
5   Remaja    13      11-14          trapesium_up        0.667   
6   Remaja    16      14-19          trapesium_down      0.600   
7   Pemuda    18      15-20          trapesium_up        0.600   
8   Pemuda    22      20-24          trapesium_down      0.500   
9   Dewasa    35      20-40          trapesium_up        0.750   
10  Dewasa    60      40-65          trapesium_down      0.200   
11  Lansia    80      65-85          trapesium_up        0.75